# Pandas 学习笔记

这个 notebook 用来系统学习 pandas 基础。它的风格和 `numpy-study.ipynb` 保持一致：先用 Markdown 说明概念，再用小段代码练习。

pandas 最适合**处理表格数据**，例如 CSV、Excel、数据库查询结果、业务报表等。


## 一、导入 pandas 和准备数据

pandas 通常使用 `pd` 作为别名。后面的示例会先构造一份学生成绩数据，用它来练习选择、筛选、统计、清洗和合并。


In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_columns", 20)


In [ ]:
students = pd.DataFrame({
    "name": ["张三", "李四", "王五", "赵六", "钱七", "孙八"],
    "class": ["A", "A", "B", "B", "A", "B"],
    "math": [92, 78, 85, 60, np.nan, 88],
    "english": [81, 86, np.nan, 75, 90, 83],
    "city": ["北京", "上海", "北京", "深圳", "上海", "深圳"],
    "join_date": ["2024-01-10", "2024-01-12", "2024-02-05", "2024-02-20", "2024-03-01", "2024-03-12"],
})

students


## 二、Series 基础

`Series` 是 pandas 的一维数据结构，可以理解为“带索引的一列数据”。


### 2.1 创建 Series


In [ ]:
scores = pd.Series([90, 85, 72, 96], name="score")
print(scores)
print("类型:", type(scores))


### 2.2 自定义索引

Series 的索引不一定是 0、1、2，也可以使用有业务含义的标签。


In [ ]:
score_by_name = pd.Series([90, 85, 72, 96], index=["张三", "李四", "王五", "赵六"])
print(score_by_name)
print("张三的分数:", score_by_name["张三"])


## 三、DataFrame 基础

`DataFrame` 是 pandas 最常用的二维表格结构，可以理解为“带行索引和列名的表”。


### 3.1 创建 DataFrame


In [ ]:
df = pd.DataFrame({
    "product": ["apple", "banana", "orange"],
    "price": [5.5, 3.2, 4.8],
    "stock": [100, 80, 120],
})

df


### 3.2 查看 DataFrame 的基本信息

常用属性：

- `shape`：行数和列数。
- `columns`：列名。
- `index`：行索引。
- `dtypes`：每列数据类型。


In [ ]:
print("shape:", students.shape)
print("columns:", students.columns.tolist())
print("index:", students.index.tolist())
print("dtypes:")
print(students.dtypes)


## 四、快速查看数据

拿到一份表格数据后，先不要急着分析，应该先看数据长什么样、有哪些列、是否有缺失值。


### 4.1 head() 和 tail()


In [ ]:
print("前 3 行:")
print(students.head(3))
print("后 2 行:")
print(students.tail(2))


### 4.2 info() 和 describe()

`info()` 看数据类型和缺失情况，`describe()` 看数值列的统计摘要。


In [ ]:
students.info()


In [ ]:
students.describe()


## 五、选择数据

pandas 选择数据的方式很多，初学时建议先掌握四种：选列、选多列、`loc`、`iloc`。


### 5.1 选列


In [ ]:
print(students["name"])
print(type(students["name"]))


### 5.2 选多列


In [ ]:
students[["name", "math", "english"]]


### 5.3 loc 按标签选择

`loc` 使用行标签和列名选择数据。


In [ ]:
students.loc[0:2, ["name", "class", "math"]]


### 5.4 iloc 按位置选择

`iloc` 使用行号和列号选择数据，和 NumPy 的位置索引更像。


In [ ]:
students.iloc[0:3, 0:3]


## 六、布尔筛选

布尔筛选是 pandas 中最常用的操作之一，它和 NumPy 的布尔索引思想一致。


### 6.1 单条件筛选


In [ ]:
students[students["math"] >= 85]


### 6.2 多条件筛选

多条件筛选时，每个条件要用括号包起来，条件之间用 `&` 或 `|` 连接。


In [ ]:
students[(students["class"] == "A") & (students["english"] >= 85)]


### 6.3 isin() 多值匹配


In [ ]:
students[students["city"].isin(["北京", "上海"])]


## 七、缺失值处理

pandas 中常用 `isna()`、`notna()`、`dropna()`、`fillna()` 处理缺失值。


### 7.1 检查缺失值


In [ ]:
print(students.isna())
print("每列缺失值数量:")
print(students.isna().sum())


### 7.2 删除缺失值


In [ ]:
students.dropna()


### 7.3 填充缺失值

数值列常用平均值、中位数或 0 填充。


In [ ]:
students_filled = students.copy()
students_filled["math"] = students_filled["math"].fillna(students_filled["math"].mean())
students_filled["english"] = students_filled["english"].fillna(students_filled["english"].mean())

students_filled


## 八、新增、修改和删除列

DataFrame 可以像字典一样新增或修改列。实际数据分析里，创建新指标非常常见。


### 8.1 新增列


In [ ]:
students2 = students_filled.copy()
students2["total"] = students2["math"] + students2["english"]
students2["avg"] = students2[["math", "english"]].mean(axis=1)

students2


### 8.2 按条件新增列


In [ ]:
students2["level"] = np.where(students2["avg"] >= 85, "优秀", "普通")
students2[["name", "avg", "level"]]


### 8.3 删除列


In [ ]:
students2.drop(columns=["total"]).head()


## 九、排序、去重和索引处理

这些是日常数据清洗和查看结果时很常用的操作。


### 9.1 sort_values() 排序


In [ ]:
students2.sort_values("avg", ascending=False)


### 9.2 drop_duplicates() 去重


In [ ]:
city_df = pd.DataFrame({"city": ["北京", "上海", "北京", "深圳"]})
city_df.drop_duplicates()


### 9.3 set_index() 和 reset_index()


In [ ]:
name_indexed = students2.set_index("name")
print(name_indexed.head())
print("恢复普通列:")
print(name_indexed.reset_index().head())


## 十、字符串和日期处理

pandas 对字符串和日期提供了很好用的批量处理能力。


### 10.1 str 字符串方法


In [ ]:
text_df = pd.DataFrame({"raw_name": [" alice ", "BOB", " Cathy"]})
text_df["clean_name"] = text_df["raw_name"].str.strip().str.title()
text_df


### 10.2 to_datetime() 日期转换


In [ ]:
students_time = students2.copy()
students_time["join_date"] = pd.to_datetime(students_time["join_date"])
students_time["join_month"] = students_time["join_date"].dt.month
students_time[["name", "join_date", "join_month"]]


## 十一、统计和分组

分组统计是 pandas 的重点。常见问题包括：每个班平均分是多少？每个城市有多少人？每类商品销售额是多少？


### 11.1 value_counts() 计数


In [ ]:
students2["city"].value_counts()


### 11.2 groupby() 分组统计


In [ ]:
students2.groupby("class")[["math", "english", "avg"]].mean()


### 11.3 agg() 多个统计指标


In [ ]:
students2.groupby("class").agg(
    student_count=("name", "count"),
    math_mean=("math", "mean"),
    avg_max=("avg", "max"),
)


## 十二、合并数据

pandas 常用两类合并：

- `concat()`：按行或按列拼接。
- `merge()`：按关键列关联两张表，类似 SQL join。


### 12.1 concat() 拼接


In [ ]:
part1 = students2.iloc[:3]
part2 = students2.iloc[3:]

pd.concat([part1, part2], ignore_index=True)


### 12.2 merge() 关联


In [ ]:
class_info = pd.DataFrame({
    "class": ["A", "B"],
    "teacher": ["陈老师", "周老师"],
})

students2.merge(class_info, on="class", how="left")


## 十三、文件读写

pandas 最常见的入口是 `read_csv()` 和 `read_excel()`，常见导出方式是 `to_csv()` 和 `to_excel()`。这里用 CSV 演示一个完整读写流程。


### 13.1 to_csv() 导出 CSV


In [ ]:
output_path = "students_demo.csv"
students2.to_csv(output_path, index=False, encoding="utf-8-sig")
print("已导出:", output_path)


### 13.2 read_csv() 读取 CSV


In [ ]:
students_from_csv = pd.read_csv("students_demo.csv")
students_from_csv.head()


## 十四、学习小结

这份 notebook 先要掌握的主线是：

1. 认识 `Series` 和 `DataFrame`。
2. 学会查看表格基本信息。
3. 熟练使用 `loc`、`iloc` 和布尔筛选。
4. 掌握缺失值处理和新增列。
5. 学会 `groupby()` 分组统计。
6. 理解 `concat()` 和 `merge()` 的差别。
7. 能读写 CSV 文件。

后续可以继续学习更进阶的内容：透视表 `pivot_table()`、时间序列、窗口函数、大文件分块读取、和数据可视化。
